# Notebook 3: End-To-End Desktop System Walkthrough

This final notebook runs the whole Phase 1 desktop mock as a system:

1. Generate a clean edge database from the bundled example audio.
2. Start the local FastAPI hub service.
3. Send pending edge rows over HTTP with MessagePack.
4. Confirm the hub database received the batch.
5. Run the watchdog.
6. Write an end-to-end report.

Unlike Notebooks 1 and 2, this notebook intentionally uses the production-style scripts and network path. It is still a desktop mock, but the control flow looks like the real system.

## System Flow

```mermaid
sequenceDiagram
    participant Audio as example_audio WAV
    participant Edge as Edge capture loop
    participant EdgeDB as Edge SQLite
    participant Sender as sender_daemon.py
    participant API as FastAPI ingest hub
    participant HubDB as Hub SQLite
    participant Watchdog as watchdog_alert.py

    Audio->>Edge: stream 15-second buffers
    Edge->>EdgeDB: buffer_events + embeddings + retained clip rows
    Sender->>EdgeDB: query sync_status='pending'
    Sender->>API: POST MessagePack batch
    API->>HubDB: transaction insert batch/events/vectors/health
    API-->>Sender: accepted_buffer_ids
    Sender->>EdgeDB: mark accepted rows synced
    Watchdog->>HubDB: read latest health_metrics
    Watchdog-->>Watchdog: healthy / stale / missing
```

In [ ]:
from pathlib import Path
import copy
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import time

import pandas as pd
import requests
import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "edge_node_mock").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_colwidth", 120)
print("Repository root:", REPO_ROOT)

In [ ]:
from mock_common.config import load_config
from edge_node_mock.src.bio_capture_loop import load_sqlite_vec, run_capture_loop
from edge_node_mock.src.init_edge_db import init_edge_db
from edge_node_mock.src.sender_daemon import send_pending_batch
from central_hub_mock.src.init_master_db import init_master_db
from central_hub_mock.src.watchdog_alert import check_watchdog

## 1. Create Clean End-To-End Output Folders

The system rehearsal uses its own output directory so it does not overwrite the edge and hub artifacts from the earlier notebooks.

In [ ]:
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "output"
E2E_OUTPUT = OUTPUT_ROOT / "e2e"
if E2E_OUTPUT.exists():
    shutil.rmtree(E2E_OUTPUT)
(E2E_OUTPUT / "edge" / "retained_audio").mkdir(parents=True, exist_ok=True)
(E2E_OUTPUT / "hub").mkdir(parents=True, exist_ok=True)

EXAMPLE_AUDIO_PATH = REPO_ROOT / "notebooks" / "example_audio" / "example1_120s_petrel.wav"
E2E_PORT = 8010
print("E2E output:", E2E_OUTPUT)
print("Example audio:", EXAMPLE_AUDIO_PATH)

## 2. Write Edge And Hub Configs

These configs are generated from the example templates. They point at notebook output paths and use a notebook-only API key.

In [ ]:
edge_config = copy.deepcopy(load_config(REPO_ROOT / "edge_node_mock" / "config" / "edge_config.example.yaml"))
edge_config.update(
    {
        "raw_audio_mount": str(EXAMPLE_AUDIO_PATH.parent),
        "raw_audio_glob": EXAMPLE_AUDIO_PATH.name,
        "edge_db_path": str(E2E_OUTPUT / "edge" / "edge_e2e.sqlite"),
        "retained_audio_dir": str(E2E_OUTPUT / "edge" / "retained_audio"),
        "hub_ingest_url": f"http://127.0.0.1:{E2E_PORT}/ingest_batch",
        "api_key": "notebook-e2e-key",
        "include_partial_final_buffer": True,
    }
)
hub_config = copy.deepcopy(load_config(REPO_ROOT / "central_hub_mock" / "config" / "hub_config.example.yaml"))
hub_config.update(
    {
        "master_db_path": str(E2E_OUTPUT / "hub" / "hub_e2e.sqlite"),
        "api_key": "notebook-e2e-key",
        "allowed_device_ids": ["pi_01"],
    }
)

EDGE_CONFIG_PATH = E2E_OUTPUT / "edge" / "edge_config.e2e.yaml"
HUB_CONFIG_PATH = E2E_OUTPUT / "hub" / "hub_config.e2e.yaml"
EDGE_CONFIG_PATH.write_text(yaml.safe_dump(edge_config, sort_keys=False), encoding="utf-8")
HUB_CONFIG_PATH.write_text(yaml.safe_dump(hub_config, sort_keys=False), encoding="utf-8")

print("Edge config:", EDGE_CONFIG_PATH)
print("Hub config:", HUB_CONFIG_PATH)

## 3. Initialize Fresh Databases And Run Edge Capture

This calls `run_capture_loop()`, the same logic behind the command-line script. Because the generated config enables `include_partial_final_buffer`, the whole teaching clip becomes eight buffers.

In [ ]:
init_edge_db(EDGE_CONFIG_PATH, reset=True)
init_master_db(HUB_CONFIG_PATH, reset=True)

capture_summary = run_capture_loop(EDGE_CONFIG_PATH)
capture_summary

## 4. Start The Hub API As A Local Service

For the full system path we start `uvicorn` in a subprocess, then the sender talks to it over HTTP.

The `finally` cell later shuts the server down. If a previous interrupted run leaves the port busy, restart the notebook kernel or change `E2E_PORT` above.

In [ ]:
env = os.environ.copy()
env["HUB_CONFIG"] = str(HUB_CONFIG_PATH)
server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "central_hub_mock.src.ingestion_api:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(E2E_PORT),
    ],
    cwd=REPO_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for attempt in range(40):
    try:
        response = requests.get(f"http://127.0.0.1:{E2E_PORT}/docs", timeout=0.25)
        if response.status_code == 200:
            break
    except requests.RequestException:
        time.sleep(0.25)
else:
    server.terminate()
    raise RuntimeError("Hub API did not start")

print("Hub API running on", f"http://127.0.0.1:{E2E_PORT}")

## 5. Send Pending Edge Rows

`send_pending_batch()` reads pending rows from the edge database, packs binary embeddings into MessagePack, posts to the API, and marks accepted buffers as `synced`.

In [ ]:
try:
    dry_run = send_pending_batch(EDGE_CONFIG_PATH, limit=8, dry_run=True)
    send_result = send_pending_batch(EDGE_CONFIG_PATH, limit=8)
finally:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()

print("Dry run:", dry_run)
print("Send result:", send_result)

## 6. Run The Watchdog And Inspect Counts

A healthy watchdog means the sender telemetry arrived at the hub and is fresh.

In [ ]:
watchdog_result = check_watchdog(HUB_CONFIG_PATH)
print(watchdog_result)

In [ ]:
edge_db = Path(edge_config["edge_db_path"])
hub_db = Path(hub_config["master_db_path"])

edge_conn = sqlite3.connect(edge_db)
edge_meta = dict(edge_conn.execute("SELECT key, value FROM schema_metadata;").fetchall())
if edge_meta["vector_table"] == "perch_vectors":
    load_sqlite_vec(edge_conn)

hub_conn = sqlite3.connect(hub_db)
hub_meta = dict(hub_conn.execute("SELECT key, value FROM schema_metadata;").fetchall())
if hub_meta["vector_table"] == "hub_perch_vectors":
    load_sqlite_vec(hub_conn)

report = {
    "edge": {
        "buffer_count": edge_conn.execute("SELECT COUNT(*) FROM buffer_events;").fetchone()[0],
        "pending_count": edge_conn.execute("SELECT COUNT(*) FROM buffer_events WHERE sync_status='pending';").fetchone()[0],
        "synced_count": edge_conn.execute("SELECT COUNT(*) FROM buffer_events WHERE sync_status='synced';").fetchone()[0],
        "embedding_segments": edge_conn.execute("SELECT COUNT(*) FROM embedding_segments;").fetchone()[0],
        "vectors": edge_conn.execute(f"SELECT COUNT(*) FROM {edge_meta['vector_table']};").fetchone()[0],
        "retained_audio_clips": edge_conn.execute("SELECT COUNT(*) FROM retained_audio_clips;").fetchone()[0],
        "retained_flac": len(list((E2E_OUTPUT / "edge" / "retained_audio").glob("*.flac"))),
    },
    "hub": {
        "batches": hub_conn.execute("SELECT COUNT(*) FROM ingestion_batches;").fetchone()[0],
        "buffers": hub_conn.execute("SELECT COUNT(*) FROM hub_buffer_events;").fetchone()[0],
        "retained_audio_clips": hub_conn.execute("SELECT COUNT(*) FROM hub_retained_audio_clips;").fetchone()[0],
        "embedding_segments": hub_conn.execute("SELECT COUNT(*) FROM hub_embedding_segments;").fetchone()[0],
        "vectors": hub_conn.execute(f"SELECT COUNT(*) FROM {hub_meta['vector_table']};").fetchone()[0],
        "health_metrics": hub_conn.execute("SELECT COUNT(*) FROM health_metrics;").fetchone()[0],
    },
    "transport": {
        "dry_run_payload_bytes": dry_run.payload_bytes,
        "sent_payload_bytes": send_result.payload_bytes,
        "accepted_buffer_ids": send_result.accepted_buffer_ids,
    },
    "watchdog": {
        "status": watchdog_result.status,
        "message": watchdog_result.message,
    },
}
report

## 7. Save The End-To-End Report

This mirrors the report created during Phase 1 implementation, but it is generated from the notebook's own disposable output directory.

In [ ]:
REPORT_PATH = E2E_OUTPUT / "e2e_report.json"
REPORT_PATH.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("Report:", REPORT_PATH)
print(json.dumps(report, indent=2))

## What You Should See

After running all three notebooks in order, inspect:

```text
notebooks/output/edge/edge_notebook.sqlite
notebooks/output/transport/payload.msgpack
notebooks/output/hub/hub_notebook.sqlite
notebooks/output/e2e/e2e_report.json
```

That set of files tells the whole story: raw audio was processed at the edge, embeddings and metadata were packaged, the hub accepted the batch, telemetry arrived, and the watchdog saw the device as healthy.